# Najafi et al. (2020) on DANDI:000016, notebook 3: all-neuron decoders at every bin

**Paper.** Najafi F, Elsayed GF, Cao R, Pnevmatikakis E, Latham PE, Cunningham JP, Churchland AK (2020).
Excitatory and inhibitory subnetworks are equally selective during decision-making and emerge
simultaneously during learning. *Neuron* 105, 165-179. [doi:10.1016/j.neuron.2019.09.045](https://doi.org/10.1016/j.neuron.2019.09.045)

**Dataset.** [DANDI:000016](https://dandiarchive.org/dandiset/000016/0.260915.2309), version
0.260915.2309 (doi:10.48324/dandi.000016/0.260915.2309), streamed from the archive. This is the only
published version (2026-09-15); the paper cites no DANDI version (it names the NWB deposit at the
CSHL repository, doi:10.14224/1.37693, from which the dandiset is derived), and the draft holds
the same 135 files.

**Claims.** `claims/000016/10.1016-j.neuron.2019.09.045.yaml`, written from the accepted author
manuscript in PubMed Central (PMC6952547) with its supplement, and the paper's code
(github.com/farznaj/imaging_decisionMaking_exc_inh at commit b7352acd). The claims file decides what
is tested and how; this notebook does not change a claim.

Claims tested here (the full table is in notebook 1):

| Claim | Finding | Notebook | Status |
|---|---|---|---|
| `fig3b-decoding-time-course` | All-neuron accuracy grows and peaks at the choice | 03 | tested |
| `fig3df-weights-similar` | Matched decoder weights of E and I | 02 (bin before choice), 03 (every bin) | tested |

The all-neuron decoder is trained and tested on correct trials at every 97 ms bin of the
choice-aligned traces, with C chosen separately at each bin (claims file). fig3df is judged at the
bin before the choice in notebook 2; its time course is shown here.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import najafi as nj
import sessions as S

pd.set_option("display.width", 200)
MICE = [1, 2, 3, 4]
A = nj.asset_urls()
print(f"DANDI:{nj.DANDISET_ID} version {nj.DANDISET_VERSION}: {len(A)} NWB files")
items = [(i, p, u) for i, (p, u) in enumerate(zip(A.path, A.url))]

R = nj.map_sessions(S.decode_time_course, items, workers=4)
OK = [r for r in R if r["ok"]]
binw = np.mean([np.diff(r["bin_times"]).mean() for r in OK])
print(len(R), "sessions loaded,", len(OK), "used; bin width %.1f ms" % binw)

## Data check

In [ ]:
d = pd.DataFrame([dict(mouse=r["mouse"], ok=r["ok"], bins=len(r["bin_times"]), first=r["bin_times"][0], last=r["bin_times"][-1]) for r in R])
print(d.groupby("mouse").agg(sessions=("ok", "size"), used=("ok", "sum"), bins_min=("bins", "min"), bins_max=("bins", "max"),
                             first_bin_ms_max=("first", "max"), last_bin_ms_min=("last", "min")).round(1).to_string())
acc = np.concatenate([r["acc_all"] for r in OK]); sh = np.concatenate([r["shfl_all"] for r in OK])
print("Accuracy over all sessions and bins: %.1f to %.1f%%; shuffle %.1f to %.1f%%" % (acc.min(), acc.max(), sh.min(), sh.max()))

## fig3b-decoding-time-course

> Accuracy of the all-neuron choice decoder grows after stimulus onset and peaks at the time of
> the choice.

Criterion: per mouse the mean over sessions at each bin (bins all of a mouse's sessions share).
"At the time of the choice" read as (a) the peak in one of the two bins around the choice (97 ms
before to 97 ms after) and (b) within 300 ms of the choice (bins starting from -291 to +291 ms).
Reproduced when in every mouse the earliest bin is lower than the bin before the choice and the peak
lies in the window.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(15, 3), sharey=True)
rows = []
for k, m in enumerate(MICE):
    rr = [r for r in OK if r["mouse"] == m]
    a, off = nj.align_bins([r["acc_all"] for r in rr], [r["i1"] for r in rr])
    s_, _ = nj.align_bins([r["shfl_all"] for r in rr], [r["i1"] for r in rr])
    mean = a.mean(0); t = off * binw; pk = int(np.argmax(mean))
    rows.append(dict(mouse=m, sessions=len(rr), first_bin_ms=round(t[0]), first=mean[0], before_choice=mean[off == -1][0],
                     peak=mean[pk], peak_bin_start_ms=round(t[pk]), rises=mean[0] < mean[off == -1][0],
                     peak_a=off[pk] in (-1, 0), peak_b=abs(t[pk]) < 300))
    ax[k].plot(t, mean, "k"); ax[k].plot(t, s_.mean(0), c="0.7"); ax[k].axvline(0, ls="--", c="gray"); ax[k].set_title(f"mouse {m}")
    print(f"mouse {m} accuracy per bin from {round(t[0])} ms:", np.round(mean, 1).tolist())
ax[0].set_ylabel("accuracy (%)"); plt.show()
T = pd.DataFrame(rows)
print(T.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("Reading (a) holds in every mouse:", bool((T.rises & T.peak_a).all()), "; reading (b):", bool((T.rises & T.peak_b).all()))

**Result: (to be written)**

## fig3df-weights-similar, time course

Mean |w| of excitatory and inhibitory neurons in the unit-length all-neuron weight vector at every
bin; unpaired t-test across sessions per mouse in each bin. The outcome is judged at the bin
before the choice in notebook 2.

In [ ]:
for m in MICE:
    rr = [r for r in OK if r["mouse"] == m]
    we, off = nj.align_bins([r["wabs_exc"] for r in rr], [r["i1"] for r in rr])
    wi, _ = nj.align_bins([r["wabs_inh"] for r in rr], [r["i1"] for r in rr])
    p = np.array([stats.ttest_ind(we[:, k], wi[:, k]).pvalue for k in range(len(off))])
    print(f"mouse {m}: {len(off)} bins; mean |w| exc {we.mean():.4f}, inh {wi.mean():.4f}; bins with p < 0.05: "
          f"{int((p < 0.05).sum())} (offsets {off[p < 0.05].tolist()}); inh above exc in {int((wi.mean(0) > we.mean(0)).sum())} bins")

## Summary

(to be written)